# AI-Sec Inbound Lab — a real prompt-security edge, proven by real attacks

A single-node AI prompt-security lab you can run on one Apple-Silicon
MacBook: a real Kubernetes edge, a real policy gate, a real LLM
application — and a real attack that gets through when the gate is
removed, then gets stopped when it isn't.

## What this lab achieves

Security demos usually stop at "the filter blocks bad strings". This lab
proves the whole loop end to end:

1. **The vulnerability is real.** Behind the gate sits Atlas, a
   deliberately vulnerable customer-support assistant running a genuine
   LLM. Asked about order 1002, Atlas reads an attacker-controlled
   "support note" as instructions and leaks its (fake) canary API key
   into a CRM sink — on that gate-free path, the leak is proven by
   assertion against the sink's log.
2. **The defense is layered, cheap-first.** Every prompt at the edge is
   judged by NOVA's four evaluator tiers — keywords (<1 ms), semantics
   (~15 ms), an LLM judge (~200 ms–2 s), then the Laya decision model
   (~40 ms CPU) — cheap tiers short-circuiting expensive ones.
3. **The gate holds.** The same attacks through the edge all return 403,
   the CRM sink gains nothing, and every decision is attributed in the
   audit log.
4. **The gate is transparent.** Benign prompts pass with real model
   answers (HTTP 200) — a gate that blocked everything would also pass
   every attack test.

Everything is open source and declarative: the machine is a NixOS flake,
the cluster is OpenTofu, the policy is `.nov` rule files in Git, and the
lab's acceptance criteria are the notebooks themselves (`nbdev_test`).

## How a request flows

```
curl ──▶ NGINX Gateway Fabric (HTTPRoute, :80)
          └─▶ nova-gate pod
                ├─ NOVA: keywords → semantics → LLM judge   (fail closed on any hit → 403 JSON)
                └─ Laya: typed verdict (injection / jailbreak / benign)
                      └─▶ Atlas pod ──▶ real LLM answer (200)
```

Atlas is also reachable from *inside* the cluster (`atlas:8080`) without
the gate — that unprotected path is what makes the proof in
`05_attacks.ipynb` possible: first show the attack really works, then show
the gate stopping it.

**The components.**

| Component | What it does | Where it lives |
|---|---|---|
| NGINX Gateway Fabric | Gateway API edge — routes inbound `/v1/*` | k3s (NGF controller) |
| NOVA | scans every prompt, 4 evaluator tiers, fail-closed | `nova-gate` pod |
| Laya | 421M open-weights decision model: injection / jailbreak / benign | `nova-gate` pod |
| Atlas | deliberately vulnerable support assistant — real LLM, tool loop, CRM sink | `atlas` pod |
| Ollama | hosts the models at native Metal speed; the one host↔VM seam | macOS host |
| OpenTofu | declares all ten cluster objects; `apply` = converged | runs in the VM |
| NixOS flake | declares the machine: k3s, docker, toolchain | the VM |

## How to click through it

Open the notebooks in order (`00_core` → `01_nova_rules` → `02_gate_app`
→ `03_victim` → `04_deploy` → `05_attacks`). Every cell narrates why it
exists, what it does, and what success looks like before you run it, and
each notebook ends by pointing at the next.

## Notebooks

| Notebook | What it is |
|---|---|
| `00_core.ipynb` | constants + cluster connectivity (the front-door test) |
| `01_nova_rules.ipynb` | the `.nov` policy — one rule per evaluator tier, test-scanned |
| `02_gate_app.ipynb` | the gate (NOVA + Laya) + its container image |
| `03_victim.ipynb` | Atlas — the vulnerable app and its three weaknesses |
| `04_deploy.ipynb` | OpenTofu: validate → apply → rollouts → first edge probe |
| `05_attacks.ipynb` | the two-path proof: real leak (direct), real block (edge) |

Run: `nbdev_test --n_workers 0`. Publish: `nbdev_docs`.